# AmazingHand — Kéo Búa Bao với model RPS H5 và Laya local

Chạy Cell 1–8 một lần; Cell 9 mở ứng dụng game trong cửa sổ 1920×1080, có nút bắt đầu, tiếp tục, reset người chơi và thoát. Model nhận diện là `models/rps.h5`, được load một lần. Laya và cấu hình motor giữ nguyên. Không dùng MediaPipe, không train thêm, không gọi API inference.

Đối chiếu `training-hand.ipynb`: ảnh RGB 150×150, resize nearest, chia 255; `flow_from_directory` không truyền `classes` nên nhãn alphabet là **paper=0, rock=1, scissors=2**. [Tài liệu Keras](https://keras.io/getting_started/) và [load_model](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/).

**Giới hạn model:** đây là classifier 3 lớp, không có lớp không-tay/khác và không xuất landmarks. Confidence cao không bảo đảm có tay hoặc chỉ có một tay. Hãy đặt đúng một bàn tay trong ROI, nền đơn giản giống ảnh train; ảnh quá phẳng/tối, confidence hoặc margin thấp trả UNKNOWN. Không thể bảo đảm loại mọi nền/gesture lạ bằng model này.


**Luồng chụp ảnh:** countdown 3 giây → RA TAY (gửi lệnh robot) → chụp một frame → H5 phân loại đúng một lần → chờ robot xác nhận → chấm điểm → thêm lịch sử cho Laya ở ván sau. Ván đầu Laya tự chọn với lịch sử rỗng. Không nhận diện realtime trong trận.

**Máy mới:** clone/pull repo, chọn kernel Python 3.12 và chạy các cell. Cell 1 tự cài thư viện thiếu hoặc sai phiên bản vào đúng kernel; chỉ cần restart nếu package đã được kernel load bị thay phiên bản. `rps.h5` đi kèm repo và được kiểm tra SHA-256; nếu thiếu, Cell 3 tự tải lại. Laya tự tải khoảng 808 MB ở Cell 6 lần đầu, các lần sau dùng local. Lần đầu cần Internet. Python, VS Code/Jupyter và driver USB/camera phải có sẵn để mở notebook; không thể tự tạo kernel từ một notebook chưa chạy. Cell 2 tự chọn cổng USB serial duy nhất; sửa PORT nếu có nhiều thiết bị, hoặc ROBOT_ENABLED=False khi chỉ test camera.


In [ ]:
from pathlib import Path
import os
import sys
import platform
import importlib
import importlib.util
from importlib.metadata import version, PackageNotFoundError
import atexit
import shlex

os.environ["KERAS_BACKEND"] = "torch"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "PythonExample" / "AmazingHand_RPSGame.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("FAIL — Mở notebook từ thư mục AmazingHand hoặc PythonExample.")
helper_path = str(PROJECT_ROOT / "PythonExample")
if helper_path not in sys.path:
    sys.path.insert(0, helper_path)

# Đóng phiên trước trước khi tạo tài nguyên khác; không để trùng cổng COM/webcam.
if "close_session" in globals():
    close_session()
resources = {}
def close_session():
    errors = []
    for key in ("game", "robot", "camera", "recognizer", "decision_engine", "display"):
        resource = resources.pop(key, None)
        if resource is not None:
            try:
                resource.close()
            except Exception as exc:
                errors.append(f"{key}: {exc}")
    if errors:
        print("Lỗi giải phóng tài nguyên:", errors)
if "_rps_exit_callback" in globals():
    atexit.unregister(_rps_exit_callback)
_rps_exit_callback = close_session
atexit.register(_rps_exit_callback)

print("Python:", platform.python_version(), "—", sys.executable)
print("Hệ điều hành:", platform.platform())
print("PASS" if sys.version_info >= (3, 11) else "FAIL", "— Python 3.11–3.12; kernel khuyến nghị 3.12")
requirements = {
    "numpy": "numpy==1.26.4", "scipy": "scipy==1.16.1",
    "keras": "keras==3.15.1", "h5py": "h5py>=3.11,<4", "cv2": "opencv-contrib-python==4.11.0.86",
    "torch": "torch>=2", "transformers": "transformers==4.57.6",
    "laya": "laya==0.4.3", "huggingface_hub": "huggingface_hub>=0.20",
    "safetensors": "safetensors>=0.4", "rustypot": "rustypot==1.11.0",
    "serial": "pyserial>=3.5,<4", "IPython": "ipython",
    "ipywidgets": "ipywidgets>=8,<9", "jupyter_ui_poll": "jupyter-ui-poll>=0.2,<1",
}
from AmazingHand_Setup import ensure_dependencies
ensure_dependencies(PROJECT_ROOT / "requirements-rps-game.txt")
missing = []
environment_errors = ["Package chưa cài: " + item for item in missing]
environment_ok = not missing
for module in requirements:
    try:
        imported = importlib.import_module(module)
        print("PASS —", module, getattr(imported, "__version__", "đã import"))
    except Exception as exc:
        environment_ok = False
        environment_errors.append(f"{module}: {type(exc).__name__}: {exc}")
        print("FAIL —", module, type(exc).__name__, str(exc))

if environment_ok:
    import torch
    torch.set_num_threads(min(8, os.cpu_count() or 1))
    from transformers.modeling_utils import no_init_weights
    import keras
    print("PASS — CPU; CUDA:", torch.cuda.is_available())
    if keras.backend.backend() != "torch":
        raise RuntimeError("Keras backend chưa đúng. Restart Kernel rồi chạy lại Cell 1.")
    print("PASS — Keras/PyTorch; model RPS:", (PROJECT_ROOT / "models/rps.h5").is_file())
    for name in ("AmazingHand_HardwareDemo", "AmazingHand_Motion", "AmazingHand_GripCycles",
                 "AmazingHand_VSign", "AmazingHand_Notebook", "AmazingHand_LayaRPS", "AmazingHand_RPSGame"):
        try:
            module = importlib.import_module(name)
            importlib.reload(module)
            print("PASS —", name)
        except Exception as exc:
            environment_ok = False
            environment_errors.append(f"{name}: {type(exc).__name__}: {exc}")
            print("FAIL —", name, str(exc))
    import cv2
    probe = cv2.VideoCapture(0, cv2.CAP_DSHOW if os.name == "nt" else cv2.CAP_ANY)
    try:
        ok, _ = probe.read() if probe.isOpened() else (False, None)
        print("PASS" if ok else "FAIL", "— Camera 0; có thể đổi camera_id tại Cell 2.")
    finally:
        probe.release()
    from AmazingHand_RPSGame import GameNotReady
    game = GameNotReady()
if not environment_ok:
    raise RuntimeError("FAIL — Lỗi dependency/module:\n" + "\n".join(environment_errors) + "\nKernel: " + sys.executable)
print("PASS — Môi trường sẵn sàng. Tiếp theo: Cell 2.")

## Cell 2 — Cấu hình

**Đổi `robot_enabled=False` để thử camera và luật chơi mà không mở cổng serial.** ROI mặc định ở bên trái ảnh; đặt AmazingHand bên phải, ngoài khung xanh. Chỉnh `roi=(trái, trên, phải, dưới)` theo tỷ lệ 0–1 rồi chạy lại Cell 4–8 nếu cần đổi góc nhìn.

Motor giữ đúng cấu hình notebook RPS hiện có: 80°/s, điểm duỗi `(70, 60, 37, 35)`, điểm co `(92, 90, 90, 90)`, không ghi EEPROM. `reset_pose()` dùng tư thế Paper đã hiệu chỉnh rồi nhả torque. Không tăng góc hoặc tốc độ tự động.

`display_backend="notebook"` nếu cửa sổ OpenCV không phù hợp với máy bạn. Khi dùng notebook, nhấn Interrupt để dừng; OpenCV có phím Q/Esc. Preview tự kết thúc sau 12 giây.

In [ ]:
from AmazingHand_RPSGame import GameConfig
from AmazingHand_Setup import choose_serial_port

ROBOT_ENABLED = True  # False: test camera/game trên máy không có bàn tay.
PORT = "auto"  # Hoặc COM3, COM4, /dev/ttyUSB0... nếu có nhiều USB serial.

config = GameConfig(
    camera_id=0,
    camera_width=1280,
    camera_height=720,
    camera_fps=30,
    window_width=1920,
    window_height=1080,
    target_wins=3,
    show_confirmation_buttons=False,  # Ẩn nút; vẫn xác nhận bằng phím 1/2/3.
    laya_exploration=0.30,  # 70% phân bố Laya + 30% phân bố đều; tăng để đa dạng hơn.
    require_confirmation=True,  # Mỗi ván phải bấm 1:R, 2:P hoặc 3:S trước khi chấm điểm.
    confidence_threshold=0.75,
    smoothing_window=5,
    minimum_agreement=4,
    countdown_seconds=3,
    snapshot_delay_seconds=0.25,  # Chụp sau RA TAY 250 ms; chỉnh nếu cần thêm thời gian giơ tay.
    rps_model_path="models/rps.h5",
    rps_class_names=("paper", "rock", "scissors"),
    rps_pixel_scale=1/255,
    prediction_margin=0.15,
    minimum_roi_std=8,
    robot_enabled=ROBOT_ENABLED,
    roi=(0.03, 0.05, 0.58, 0.95),
    preview_seconds=12,
    display_backend="auto",
    port=choose_serial_port(PORT) if ROBOT_ENABLED else PORT,
    speed_deg_s=160,  # Độ/giây; servo vẫn theo phản hồi tải và vị trí.
    reset_move="rock",  # Tư thế chờ: co cả 4 ngón.
    open_extensions=(70, 60, 37, 35),
    close_flexions=(92, 90, 90, 90),
)
config.validate()
print("PASS — Cấu hình:", config)

## Cell 3 — Load model RPS local một lần

Đọc H5 bằng Keras 3 backend PyTorch, `compile=False`; kiểm tra input/output và warmup. Không tải model và không khởi tạo lại giữa các ván.

In [ ]:
from AmazingHand_RPSGame import RPSModelSession
from AmazingHand_Setup import ensure_rps_model

if resources.get("recognizer") is None or resources["recognizer"].closed:
    resources["recognizer"] = RPSModelSession(ensure_rps_model(PROJECT_ROOT, config.rps_model_path), config)
recognizer_session = resources["recognizer"]
print("PASS — RPS H5 local: RGB 150×150 /255; nhãn:", config.rps_class_names)


## Cell 4 — Camera và kiểm tra 3 tư thế

Preview có timeout, Q/Esc để thoát. Đặt đúng một tay trong ROI xanh, giữ lần lượt ROCK/PAPER/SCISSORS đến khi xác nhận. Model không xuất landmarks. Nếu nhầm nhãn hoặc nền bị nhận thành tay, dừng kiểm tra và chỉnh góc nhìn/nền/ROI trước khi chơi.

In [ ]:
from AmazingHand_RPSGame import CameraStream, GestureDetector, GameDisplay, preview_camera

if resources.get("game") is not None:
    resources.pop("game").close(keep_models=True)
if resources.get("camera") is not None:
    resources["camera"].close()
camera = CameraStream(config).open()
resources["camera"] = camera
preview_detector = GestureDetector(recognizer_session, config)
preview_display = GameDisplay(config)
preview_gestures = preview_camera(camera, preview_detector, preview_display, config.preview_seconds)
print("PASS — Camera được giữ mở; tiếp theo Cell 5.")

## Cell 5 — Phân loại ảnh đã chụp

Trong trận, mỗi ván chụp đúng một frame và gọi H5 đúng một lần: crop ROI → RGB → nearest 150×150 → float32 /255 → confidence/margin. Không dùng đồng thuận 4/5 vì chỉ có một ảnh; không nhân bản ảnh để bỏ qua điều kiện này. Ảnh không đạt ngưỡng trả UNKNOWN. Người chơi vẫn xác nhận bằng 1:R / 2:P / 3:S để chấm điểm và lưu dữ liệu cho Laya. Log giữ riêng dự đoán H5 và xác nhận của người chơi.

Riêng preview Cell 4 vẫn nhận diện liên tục để kiểm tra góc nhìn trước trận.


In [ ]:
from AmazingHand_RPSGame import GestureDetector

detector = GestureDetector(recognizer_session, config)
print("PASS — RPS H5, nhãn paper/rock/scissors, phân loại một ảnh/ván, lọc confidence/margin.")

## Cell 6 — Khởi tạo Laya và AmazingHand

Laya chỉ dự đoán **nước người chơi tiếp theo từ lịch sử hợp lệ**. Robot lấy mẫu theo phân bố nước khắc chế từ xác suất Laya, trộn 30% phân bố đều để đa dạng hơn; nước người chơi ở round hiện tại không được truyền vào Laya. Không huấn luyện thêm hoặc gọi inference API.

Adapter giữ nguyên `NotebookHand`, `run_cycles`, các kiểm tra giới hạn/điện áp/tải và xác nhận hoàn tất. Chỉ thay cơ chế sở hữu serial: mở một lần, dùng lại qua các round. Kết nối ban đầu chỉ đọc telemetry. Đóng notebook điều khiển khác nếu đang giữ COM3.

**Đa dạng nước đi:** `laya_exploration=0.30` nghĩa là phân bố chọn nước gồm 70% từ Laya và 30% đều trên R/P/S. Mỗi nước có xác suất tối thiểu 10%, kể cả khi Laya nghiêng hẳn về một dự đoán. Không ép chia đều theo từng trận, nên vẫn có thể lặp nước. Dự đoán top-1 của Laya được giữ riêng để tính prediction accuracy; log lưu cả xác suất dự đoán và phân bố chọn nước robot.

In [ ]:
import time
import torch
import laya
from AmazingHand_LayaRPS import ensure_model, MODEL_REVISION
from AmazingHand_RPSGame import LayaDecisionEngine, AmazingHandRobotAdapter

if resources.get("decision_engine") is None or resources["decision_engine"].closed:
    model_path = ensure_model(PROJECT_ROOT / "models" / "laya-english", offline=config.offline)
    device = ("cuda" if torch.cuda.is_available() else "cpu") if config.device == "auto" else config.device
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("FAIL — PyTorch chưa có CUDA; chọn config.device='cpu'.")
    if device == "cpu":
        torch.set_num_threads(min(8, os.cpu_count() or 1))
    started = time.perf_counter()
    previous_offline = os.environ.get("HF_HUB_OFFLINE")
    os.environ["HF_HUB_OFFLINE"] = "1"
    try:
        agent = laya.load(str(model_path), device=device, fast=False, compile=False)
    finally:
        if previous_offline is None:
            os.environ.pop("HF_HUB_OFFLINE", None)
        else:
            os.environ["HF_HUB_OFFLINE"] = previous_offline
    resources["decision_engine"] = LayaDecisionEngine(agent, config=config)
    print(f"PASS — Laya nạp trong {time.perf_counter()-started:.2f}s, device={agent.device}")
decision_engine = resources["decision_engine"]
if resources.get("robot") is not None:
    resources["robot"].close()
robot_controller = AmazingHandRobotAdapter(config).connect()
resources["robot"] = robot_controller
print("PASS — choose_move(history), execute_move(move), reset_pose() sẵn sàng.")

## Cell 7 — Luồng một ván

Countdown 3 giây (Laya khóa nước từ lịch sử cũ) → RA TAY (robot bắt đầu thực thi) → chụp một frame mới sau `snapshot_delay_seconds` → H5 phân loại đúng một lần → chờ motor xác nhận hoàn tất → người chơi xác nhận 1:R / 2:P / 3:S → chấm điểm → reset robot.

Ván đầu: Laya nhận history rỗng. Các ván sau: chỉ nhận lịch sử những ván hợp lệ đã hoàn tất. Không gửi ảnh hoặc nước hiện tại cho Laya trước khi khóa nước.

Robot cần thời gian chuyển động vật lý: ảnh được chụp theo tín hiệu RA TAY, không chờ robot hoàn tất mới chụp. Giữ tay trong ROI từ lúc countdown kết thúc đến khi hiện ĐÃ CHỤP. Ảnh đóng băng được hiển thị đến khi bấm tiếp tục và lưu ở `logs/snapshots/`. JSONL ghi đường dẫn ảnh, nhãn, xác suất, latency. H5 trả UNKNOWN vẫn chơi được nhờ xác nhận của người chơi. Chỉ nước đã xác nhận mới dùng chấm điểm và cập nhật history; giữ nguyên confidence model, không gán confidence giả cho xác nhận thủ công. Lỗi camera/motor hoặc interrupt dừng an toàn.


In [ ]:
from AmazingHand_RPSGame import RPSGame, GameDisplay

print("PASS — Bộ điều khiển game đã sẵn sàng. Chưa tự chạy round.")

## Cell 8 — Khởi tạo một phiên game

Cell này tạo phiên mới, history và tỷ số bắt đầu từ 0. Nếu chạy lại Cell 8, phiên trước được đóng camera/serial trước khi mở phiên mới; weights/recognizer vẫn được dùng lại. **Chạy Cell 9 để bắt đầu trận chạm 3 điểm thắng; trong trận bấm nút tiếp tục. Trong cửa sổ game bấm RESET GAME để chuyển người chơi, không cần chạy lại cell.**

Để kết thúc toàn bộ phiên: chạy `close_session()` trong một cell tạm hoặc Restart Kernel. Callback cũng giải phóng khi kernel thoát bình thường; không chương trình nào bảo đảm cleanup khi bị kill cưỡng bức.

In [ ]:
if resources.get("game") is not None:
    resources.pop("game").close(keep_models=True)
if recognizer_session.closed or decision_engine.closed:
    raise RuntimeError("Model đã đóng. Chạy lại Cell 3, 5, 6 rồi Cell 8.")
camera.open()
robot_controller.connect()
camera.get_frame()
display = GameDisplay(config)
resources["display"] = display
game = RPSGame(
    config=config, camera=camera, detector=detector,
    decision_engine=decision_engine, robot_controller=robot_controller,
    display=display, log_path=PROJECT_ROOT / config.log_path,
)
resources["game"] = game
print("GAME READY — Chạy cell cuối để bắt đầu trận chạm 3 điểm thắng.")

## Cell 9 — MỞ GAME

Chạy cell một lần để mở cửa sổ 1920×1080. Chọn cửa sổ game để nhận phím:

- **Enter:** bắt đầu / sang ván tiếp theo; sau trận thì Enter chuẩn bị người chơi mới. Không có nút tiếp tục trong cửa sổ OpenCV.
- **1: ROCK / 2: PAPER / 3: SCISSORS:** xác nhận nước; chỉ bấm phím 1/2/3; ba nút xác nhận đã ẩn.
- **RESET GAME [R]:** xóa điểm/lịch sử cho người chơi mới, giữ model và kết nối. Nút chỉ bật giữa các ván khi motor đã hoàn tất.
- **THOÁT [Q/Esc]:** đóng game, kết thúc cell. Thoát giữa chuyển động sẽ dừng an toàn; nếu phiên bị khóa thì chạy lại Cell 8.

Tư thế chờ là **ROCK: co cả 4 ngón**, thực hiện khi mở game, reset người chơi và sau mỗi ván. Tốc độ đặt tại Cell 2 là **160 độ/giây**; vẫn giữ kiểm tra tải, điện áp và xác nhận vị trí. Motor có thể tự giảm nhịp nếu chưa theo kịp.

Ai chạm 3 điểm trước thắng, hòa chơi tiếp. Log ở `logs/rps_ui_session.log`, kết quả JSONL và ảnh trong `logs/snapshots/`. Không cần chạy lại cell cho người chơi tiếp theo. Nếu OpenCV không mở được và dùng giao diện notebook dự phòng, vẫn có nút widget tương ứng Enter để thao tác.


In [ ]:
if "game" not in globals():
    print("Chạy Cell 8 để khởi tạo game trước.")
else:
    game.run_app()